<a href="https://colab.research.google.com/github/VincentUwumboriyhieGmayinaam/AI-Enabled-Depression-Screening-and-Decision-Support-for-PLHIV-in-Ghana-Using-PHQ-9/blob/main/AI_in_Healthcare_Modelling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================
# Multi-model Depression Risk Training (PLHIV)
# ============================
# - One-Hot Encoding for categoricals
# - Scaling for numeric features
# - Outcome: PHQ-9 >= 10 -> 1 else 0 (risk-score target is probability of class 1)
# - Models: Logistic, DecisionTree, RandomForest, KNN, GradientBoosting,
#           BernoulliNB (with binarizer), XGBoost*, LightGBM*, CatBoost*, MLP NN
#           (*optional if installed)
# - Outputs saved in ./outputs/
# ============================

import os, json, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler, Binarizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import BernoulliNB
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    roc_auc_score, roc_curve, confusion_matrix,
    accuracy_score, precision_recall_fscore_support
)
import joblib
import matplotlib.pyplot as plt

# Optional libraries
try:
    from xgboost import XGBClassifier
except Exception:
    XGBClassifier = None

try:
    from lightgbm import LGBMClassifier
except Exception:
    LGBMClassifier = None

try:
    from catboost import CatBoostClassifier
except Exception:
    CatBoostClassifier = None


In [ ]:
# ------------------------------
# 0) Load data
# ------------------------------
DATA_PATH = "Data CSV.csv"   # <- change if needed
df = pd.read_csv(DATA_PATH)

# ------------------------------
# 1) Define variables by Section using your actual headers
# ------------------------------

# Section A: Demographics
sectionA_demo = [
    "Age", "Sex", "Ethnicity", "Religion", "Marital status",
    "Educational level", "Occupation"
]

# Section A: HIV / Clinical
sectionA_clinical = [
    "Year of Diagnosis", "Year_Diagnosis",
    "StageofHIVclassification_", "WhatisyourCD4count_",
    "HowlonghaveyoubeenonART_",
    "Haveyoueverbeenscreenedorq_", "Wereyoueveraskedtogofora_",
    "Whatwasyourresult_", "Coughfortwo2ormoreweeks_", "Nightsweats_",
    "Chillsandfever_", "Chestpain_", "Weightloss_",
    "HaveyoubeeninitiatedonTPT_", "IfYESwhendidyouinitiate_"
]

# Section B: PHQ-9 items (PHQ-2 are first two)
phq9_items = [
    "Littleinterestorpleasureind_1",
    "Feelingdowndepressedorhope_1",
    "Troublefallingorstayingaslee_1",
    "Feelingtiredorhavinglittlee_1",
    "Poorappetiteorovereating_1",
    "Feelingbadaboutyourselfor_1",
    "Troubleconcentratingonthings_1",
    "Movingorspeakingsoslowlytha_1",
    "Thoughtsthatyouwouldbebette_1",
]

# Section C: Risk-factor domains (examples from your file)
risk_health = [
    "Doyouhaveanycoexistingphys_", "Haveyouexperiencedanymedicat_",
    "DoesHIVAIDSimpactyourdaily_1", "Arethereanyspecificphysical_1",
    "Howdoyouratetheoverallimpa_1",
]
risk_behavior = [
    "Doyouengageinsubstanceabuse_", "Doyouusetobacco_",
    "Haveyoureceivedinformationan_", "Ifyeshaveyouimplementedthe_",
    "Adheretorecommendedlifestyle",
]
risk_service = [
    "Accesstohealthcareservices_1", "Qualityofhealthcareservices_1",
    "Availabilityofcounselingandp_1", "Tailoredmentalhealthservices_1",
    "Positiveproviderpatientrelati_1", "Howcomfortabledoyoufeeldisc_1",
]
risk_cultural = [
    "Doculturalbeliefsinfluenceyo_1", "Arethereculturalnormssurroun_1",
    "Doyouthinkthereisastigma_1", "Arethereculturalbeliefsorat_1",
    "Howacceptingisyourculturalc_1", "BS_1",
]
risk_psych = [
    "Doyoutendtoworryorfeelanx_1", "Haveyouexperiencedfeelingsof_1",
    "Howoftendoyoufeellowinene_1", "Doyoufinditchallengingtoex_1",
    "Haveyouhadrecurrentthoughts_1",
    "Doyoutendtoworryorfeelanx_12", "Haveyouexperiencedfeelingsof_12",
    "Howoftendoyoufeellowinene_12", "Doyoufinditchallengingtoex_12",
    "Haveyouhadrecurrentthoughts_12",
]

sectionC = risk_health + risk_behavior + risk_service + risk_cultural + risk_psych

In [ ]:
# Final feature set (keep those that exist in the file)
FEATURES = [c for c in (sectionA_demo + sectionA_clinical + phq9_items + sectionC) if c in df.columns]

# ------------------------------
# 2) Outcome variable
# ------------------------------
if "PHQ_score" in df.columns:
    df["binary_depr"] = (df["PHQ_score"] >= 10).astype(int)
else:
    # derive total if needed
    df["phq9_sum_derived"] = df[phq9_items].sum(axis=1, skipna=True)
    df["binary_depr"] = (df["phq9_sum_derived"] >= 10).astype(int)

TARGET = "binary_depr"

# ------------------------------
# 3) Type coercion for numeric clinical fields (if accidentally stored as text)
# ------------------------------
for col in ["WhatisyourCD4count_", "HowlonghaveyoubeenonART_", "Age"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# ------------------------------
# 4) Split
# ------------------------------
X = df[FEATURES].copy()
y = df[TARGET].copy()

# Identify numeric vs categorical by dtype
num_cols = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
cat_cols = [c for c in X.columns if c not in num_cols]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

In [ ]:
# ------------------------------
# 5) Preprocessing blocks
# ------------------------------
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# handle OneHotEncoder compatibility across sklearn versions
try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", ohe)
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols),
    ],
    remainder="drop"
)

# Special pipeline for BernoulliNB: binarize post-preprocessing
bernoulli_post = Pipeline(steps=[
    ("binarize", Binarizer(threshold=0.0))
])

In [ ]:
# ------------------------------
# 6) Define models
# ------------------------------
models = {}

models["logistic"] = Pipeline([
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

models["decision_tree"] = Pipeline([
    ("prep", preprocess),
    ("clf", DecisionTreeClassifier(
        max_depth=None, random_state=42, class_weight="balanced"
    ))
])

models["random_forest"] = Pipeline([
    ("prep", preprocess),
    ("clf", RandomForestClassifier(
        n_estimators=400, random_state=42,
        class_weight="balanced_subsample", n_jobs=-1
    ))
])

models["knn"] = Pipeline([
    ("prep", preprocess),
    ("clf", KNeighborsClassifier(n_neighbors=11, weights="distance"))
])

models["gradient_boosting"] = Pipeline([
    ("prep", preprocess),
    ("clf", GradientBoostingClassifier(random_state=42))
])

# BernoulliNB (binarize features first)
models["bernoulli_nb"] = Pipeline([
    ("prep", preprocess),
    ("bin", bernoulli_post),
    ("clf", BernoulliNB())
])

# Optional: XGBoost
if XGBClassifier is not None:
    models["xgboost"] = Pipeline([
        ("prep", preprocess),
        ("clf", XGBClassifier(
            n_estimators=500, learning_rate=0.05, subsample=0.9, colsample_bytree=0.9,
            max_depth=4, reg_lambda=1.0, random_state=42, n_jobs=-1,
            eval_metric="logloss"
        ))
    ])

# Optional: LightGBM
if LGBMClassifier is not None:
    models["lightgbm"] = Pipeline([
        ("prep", preprocess),
        ("clf", LGBMClassifier(
            n_estimators=800, learning_rate=0.03, subsample=0.9, colsample_bytree=0.9,
            max_depth=-1, class_weight="balanced", random_state=42, n_jobs=-1
        ))
    ])

# Optional: CatBoost (silent fit; uses preprocessed numeric features)
if CatBoostClassifier is not None:
    models["catboost"] = Pipeline([
        ("prep", preprocess),
        ("clf", CatBoostClassifier(
            iterations=700, learning_rate=0.03, depth=6, loss_function="Logloss",
            l2_leaf_reg=3.0, random_seed=42, verbose=False
        ))
    ])

# Neural Network (MLP)
models["mlp_nn"] = Pipeline([
    ("prep", preprocess),
    ("clf", MLPClassifier(
        hidden_layer_sizes=(64, 32),
        activation="relu",
        solver="adam",
        alpha=1e-4,
        learning_rate_init=1e-3,
        max_iter=200,
        random_state=42,
        early_stopping=True,
        n_iter_no_change=10,
        validation_fraction=0.1
    ))
])

In [ ]:
# ------------------------------
# 7) Train/Evaluate helpers
# ------------------------------
os.makedirs("outputs", exist_ok=True)

def evaluate(pipe, X_tr, y_tr, X_te, y_te, model_key):
    pipe.fit(X_tr, y_tr)

    # predict_proba might not exist for some models; guard
    if hasattr(pipe, "predict_proba"):
        p_tr = pipe.predict_proba(X_tr)[:, 1]
        p_te = pipe.predict_proba(X_te)[:, 1]
    else:
        # fall back to decision_function -> prob via sigmoid
        if hasattr(pipe, "decision_function"):
            from scipy.special import expit
            p_tr = expit(pipe.decision_function(X_tr))
            p_te = expit(pipe.decision_function(X_te))
        else:
            # final fallback: use predictions as "probabilities"
            p_tr = pipe.predict(X_tr)
            p_te = pipe.predict(X_te)

    # AUC
    auc_tr = roc_auc_score(y_tr, p_tr)
    auc_te = roc_auc_score(y_te, p_te)

    # Best threshold by Youden's J
    fpr, tpr, thr = roc_curve(y_te, p_te)
    j_scores = tpr - fpr
    best_idx = int(np.argmax(j_scores))
    best_thr = thr[best_idx]
    y_pred = (p_te >= best_thr).astype(int)

    acc = accuracy_score(y_te, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_te, y_pred, average="binary")
    tn, fp, fn, tp = confusion_matrix(y_te, y_pred).ravel()
    specificity = tn / (tn + fp) if (tn + fp) else np.nan
    sensitivity = tp / (tp + fn) if (tp + fn) else np.nan

    metrics = {
        "auc_train": round(auc_tr, 4),
        "auc_test": round(auc_te, 4),
        "best_threshold": round(float(best_thr), 4),
        "accuracy": round(acc, 4),
        "sensitivity_recall": round(float(sensitivity), 4),
        "specificity": round(float(specificity), 4),
        "precision": round(float(prec), 4),
        "f1": round(float(f1), 4),
        "confusion_matrix": [[int(tn), int(fp)], [int(fn), int(tp)]]
    }

    # Save ROC
    fig, ax = plt.subplots(figsize=(5,4))
    from sklearn.metrics import RocCurveDisplay
    RocCurveDisplay.from_predictions(y_te, p_te, ax=ax, name=model_key)
    ax.set_title(f"ROC – {model_key}")
    plt.tight_layout()
    figpath = f"outputs/roc_{model_key}.png"
    plt.savefig(figpath, dpi=150)
    plt.close()

    # Save risk scores for ALL rows with this model
    all_probs = pipe.predict_proba(X)[:, 1] if hasattr(pipe, "predict_proba") else p_te
    out = df.copy()
    out[f"risk_{model_key}"] = all_probs
    out[f"risk_band_{model_key}"] = pd.cut(
        out[f"risk_{model_key}"],
        bins=[-0.01, 0.19, 0.39, 0.59, 0.79, 1.0],
        labels=["very_low", "mild", "moderate", "high", "very_high"]
    )
    out[[TARGET, f"risk_{model_key}", f"risk_band_{model_key}"]].to_csv(
        f"outputs/risk_scores_{model_key}.csv", index=False
    )

    # Save model
    joblib.dump(pipe, f"outputs/{model_key}.pkl")

    return metrics

In [ ]:
# ------------------------------
# 8) Train all, collect metrics
# ------------------------------
all_metrics = {}
for key, model in models.items():
    try:
        print(f"\nTraining: {key}")
        all_metrics[key] = evaluate(model, X_train, y_train, X_test, y_test, key)
        print(json.dumps(all_metrics[key], indent=2))
    except Exception as e:
        all_metrics[key] = {"status": "failed", "error": str(e)}
        print(f"[WARN] {key} failed: {e}")

# Save consolidated metrics
with open("outputs/metrics_all_models.json", "w") as f:
    json.dump(all_metrics, f, indent=2)

# Identify best by AUC
valid = {k:v for k,v in all_metrics.items() if "auc_test" in v}
if valid:
    best = max(valid, key=lambda k: valid[k]["auc_test"])
    print("\nBest model by AUC:", best, "→", valid[best])
else:
    print("\nNo valid models completed. Check errors in metrics_all_models.json")